# Expanded placement and recovery pilot (v4)

Local CPU simulation only: no Ray, GPU, cloud calls, Z3 or actual ML/RL training.
This pilot broadens the decisions first; learned policies will be compared later.
Read `docs/PILOT_V4.md` for the full model and limitations.

It varies active resources, sharding, placement, backups, spares, checkpoint tier,
cadence and overlap, and recovery decisions. Fixed-work comparisons preserve the
same global optimizer steps. Tensor/pipeline parallelism and multi-job scheduling
are not modeled. V3 is preserved in `PlacementPilotV3.ipynb`.


In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "learnedplacement").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "learnedplacement").is_dir(), "Open this notebook from the repository"
sys.path.insert(0, str(ROOT))

from learnedplacement.runner import read_json, status, export_run
from learnedplacement.expanded_runner import run, plot_results, validate_config
import pandas as pd


## Editable configuration

Defaults: 144 cases, 6 methods, 256 held-out traces each (221,184 evaluations),
plus planning. A full sweep can require multiple 30-minute invocations.
All profiles of a hardware instance share a train/validation/test split.
The splits are for later learning; no model is trained now.

To begin with 48 cases, set `active_nodes=[4,8]` and keep only the first two
profiles below **before starting a new output directory**. Do not change config
or source while resuming. Restart the kernel after pulling updated code.


In [ ]:
RUN_NAME = "pilot_v4"
MAX_MINUTES = 30
CONFIG = read_json(ROOT / "configs" / "pilot_v4.json")
CONFIG["active_nodes"] = [4, 8, 16]  # Reference workload size, not a fixed allocation.
CONFIG["planning_traces"] = 16
CONFIG["evaluation_traces"] = 256
CONFIG["search_candidates"] = 64
CONFIG["tuned_candidates"] = 24
CONFIG["planner_seconds"] = 20.0

# Every enabled policy dimension is visible/editable here.
CONFIG["choices"] = {
    "shards": [2, 4, 8],
    "replicas": [1, 2, 4],
    "backup_copies": [0, 1, 2],
    "spare_count": [0, 1, 2],
    "checkpoint_every": [20, 40, 80, 240],
    "backup_every": [10, 20, 80],
    "checkpoint_tier": ["remote", "local"],
    "checkpoint_mode": ["sync", "async"],
    "checkpoint_retention": [1, 2],
    "background_fraction": [0.25, 0.5],
    "recovery": ["eager", "defer"],
    "repair_every": [1, 20],
    "recovery_order": ["fastest", "smallest"],
    "source_policy": ["balanced", "first"],
}
# Optional smaller first run:
# CONFIG["active_nodes"] = [4, 8]
# CONFIG["expanded_profiles"] = CONFIG["expanded_profiles"][:2]

OUTPUT = ROOT / "results" / RUN_NAME
validate_config(CONFIG)
print("Output:", OUTPUT)
print(json.dumps(CONFIG, indent=2))


## Run or resume

Only this cell starts simulations. It saves complete observations and pauses
at the time limit. Run it again unchanged to continue. A killed process may
leave `.run.lock`; remove that file only after confirming the old process stopped.


In [ ]:
result = run(config=CONFIG, output=OUTPUT, max_minutes=MAX_MINUTES)
print(json.dumps(result, indent=2))


## Progress (read only)


In [ ]:
print(json.dumps(status(OUTPUT), indent=2))


## Saved results

Inspect censoring and budget/deadline screens alongside time. Planning latency
for `random_unaware` is attributed from the matched search, not independently
measured. These are our baselines, not reproductions of published systems.


In [ ]:
if (OUTPUT / "summary.csv").exists():
    results = pd.read_csv(OUTPUT / "summary.csv")
    display(results[["case_id", "method", "restricted_mean_s", "mean_cost_usd",
                     "censored_traces", "statistical_screen_passed", "planning_s",
                     "candidates", "active_nodes", "shards", "replicas", "spare_count",
                     "backup_copies", "checkpoint_every", "checkpoint_tier",
                     "checkpoint_mode", "recovery"]])
else:
    print("No completed comparisons yet.")


## Plots from completed cases

Dot plots use the same completed cases across methods. Censored observations
remain in restricted-time/observed-cost plots; do not interpret those as full
completion means. Choose a profile and reference size to avoid pooling assumptions.


In [ ]:
PROFILE = "reference"  # Also slow_io, slow_network, variable_repair.
REFERENCE_NODES = 8  # Also 4 or 16.
if (OUTPUT / "summary.json").exists():
    table = plot_results(OUTPUT, profile=PROFILE, reference_nodes=REFERENCE_NODES)
    display(table)
else:
    print("No saved results yet.")


## Compact analysis export

This reads saved data only. It includes cases, chosen plans, summaries, matched
pool hashes and paired differences, but excludes raw traces and candidate audits.
Raw data stays in your local run directory. An incomplete run can also be exported.


In [ ]:
archive = export_run(OUTPUT)
print("Share this file:", archive)
print("Size (MiB):", round(archive.stat().st_size / 1024**2, 3))
